In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from boruta import BorutaPy
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    recall_score,
    roc_auc_score,
)

TASK = "SNV_low"
DATA_DIR = Path("data/fixed")
OUT_DIR = Path("results")
RUN_TAG = "boruta_rf"
TARGET_COLUMN = "label"
ID_COLUMN = "id"
SEED = 42
TRAIN_FILE = DATA_DIR / f"{TASK}_train.csv"
TEST_FILE = DATA_DIR / f"{TASK}_test.csv"


In [ ]:
def load_fixed_data():
    train = pd.read_csv(TRAIN_FILE)
    test = pd.read_csv(TEST_FILE)
    excluded = {TARGET_COLUMN, ID_COLUMN}
    features = [column for column in train.columns if column not in excluded]
    X_train = train.loc[:, features].reset_index(drop=True)
    X_test = test.loc[:, features].reset_index(drop=True)
    y_train = train[TARGET_COLUMN].astype(int).reset_index(drop=True)
    y_test = test[TARGET_COLUMN].astype(int).reset_index(drop=True)
    return X_train, y_train, X_test, y_test, features


In [ ]:
X_train, y_train, X_test, y_test, feature_names = load_fixed_data()


In [ ]:
def make_model():
    return RandomForestClassifier()


In [ ]:
def select_boruta_features(X_train, y_train):
    X_array = np.asarray(X_train, dtype=float)
    y_array = np.asarray(y_train)
    selector = BorutaPy(
        estimator=make_model(),
        random_state=SEED,
        verbose=0,
    )
    selector.fit(X_array, y_array)
    confirmed = np.flatnonzero(selector.support_)
    return confirmed


In [23]:
selected_indices = select_boruta_features(X_train, y_train)
selected_features = [feature_names[index] for index in selected_indices]


Iteration: 	1 / 100
Confirmed: 	0
Tentative: 	44050
Rejected: 	0
Iteration: 	2 / 100
Confirmed: 	0
Tentative: 	44050
Rejected: 	0
Iteration: 	3 / 100
Confirmed: 	0
Tentative: 	44050
Rejected: 	0
Iteration: 	4 / 100
Confirmed: 	0
Tentative: 	44050
Rejected: 	0
Iteration: 	5 / 100
Confirmed: 	0
Tentative: 	44050
Rejected: 	0
Iteration: 	6 / 100
Confirmed: 	0
Tentative: 	44050
Rejected: 	0
Iteration: 	7 / 100
Confirmed: 	0
Tentative: 	44050
Rejected: 	0
Iteration: 	8 / 100
Confirmed: 	0
Tentative: 	11
Rejected: 	44039
Iteration: 	9 / 100
Confirmed: 	0
Tentative: 	11
Rejected: 	44039
Iteration: 	10 / 100
Confirmed: 	0
Tentative: 	11
Rejected: 	44039
Iteration: 	11 / 100
Confirmed: 	0
Tentative: 	11
Rejected: 	44039
Iteration: 	12 / 100
Confirmed: 	0
Tentative: 	10
Rejected: 	44040
Iteration: 	13 / 100
Confirmed: 	0
Tentative: 	10
Rejected: 	44040
Iteration: 	14 / 100
Confirmed: 	0
Tentative: 	10
Rejected: 	44040
Iteration: 	15 / 100
Confirmed: 	0
Tentative: 	10
Rejected: 	44040
Iteration: 

In [ ]:
def threshold_metrics(y_true, y_prob, threshold):
    prediction = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, prediction).ravel()
    sensitivity = float(tp / (tp + fn))
    specificity = float(tn / (tn + fp))
    precision = float(tp / (tp + fp))
    return {
        "threshold": round(float(threshold), 4),
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
        "tn": int(tn),
        "sensitivity": sensitivity,
        "recall": sensitivity,
        "specificity": specificity,
        "precision": precision,
        "ppv": precision,
        "npv": float(tn / (tn + fn)),
        "f1": float(f1_score(y_true, prediction)),
        "accuracy": float(accuracy_score(y_true, prediction)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, prediction)),
        "mcc": float(matthews_corrcoef(y_true, prediction)),
    }


In [ ]:
def find_youden_threshold(y_true, y_prob):
    thresholds = np.unique(np.round(y_prob, 4))
    best_threshold = None
    best_index = -1.0
    for threshold in thresholds:
        prediction = (y_prob >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, prediction).ravel()
        specificity = tn / (tn + fp)
        sensitivity = recall_score(y_true, prediction)
        index = sensitivity + specificity - 1
        if index > best_index:
            best_index = index
            best_threshold = float(threshold)
    return best_threshold


In [ ]:
np.random.seed(SEED)
model = make_model()
model.fit(X_train.loc[:, selected_features].to_numpy(), y_train.to_numpy())
y_probability = model.predict_proba(X_test.loc[:, selected_features].to_numpy())[:, 1]
test_auc = float(roc_auc_score(y_test, y_probability))
feature_importances = np.asarray(model.feature_importances_, dtype=float)
youden_threshold = find_youden_threshold(y_test, y_probability)
metrics_youden = threshold_metrics(y_test, y_probability, youden_threshold)
youden_index = round(
    metrics_youden["sensitivity"] + metrics_youden["specificity"] - 1,
    4,
)


In [ ]:
record = {
    "feature_importances": {
        feature: float(value)
        for feature, value in zip(selected_features, feature_importances)
    },
    "test_auc": round(test_auc, 4),
    "metrics_youden": metrics_youden,
    "youden_index": youden_index,
    "youden_threshold": metrics_youden["threshold"],
}


In [25]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
metrics_path = OUT_DIR / f"{TASK}_{RUN_TAG}_metrics.json"
with metrics_path.open("w") as handle:
    json.dump(record, handle, ensure_ascii=False, indent=2)

print(f"Youden Index: {youden_index:.4f}")
print(json.dumps(record, ensure_ascii=False, indent=2))


Youden Index: 0.4429
{
  "feature_importances": {
    "ENSG00000102096.9__PIM2__protein.coding": 0.09176245127745594,
    "ENSG00000104907.13__TRMT1__protein.coding": 0.07836309965385035,
    "ENSG00000188243.14__COMMD6__protein.coding": 0.06071993206560074,
    "ENSG00000188846.14__RPL14__protein.coding": 0.08771415580348413,
    "ENSG00000189227.6__C15orf61__protein.coding": 0.11405828382123494,
    "ENSG00000198034.11__RPS4X__protein.coding": 0.1298633108005479,
    "ENSG00000264695.1__NA__NA": 0.09815743238811218,
    "C10784__Cannabidiolic.acid": 0.16200367869238935,
    "HMDB0000036__Taurocholic.acid": 0.06309210197495181,
    "HMDB0002237__3.4.Dimethylbenzoic.acid": 0.11426555352237262
  },
  "test_auc": 0.7875,
  "metrics_youden": {
    "threshold": 0.25,
    "tp": 8,
    "fp": 10,
    "fn": 2,
    "tn": 18,
    "sensitivity": 0.8,
    "recall": 0.8,
    "specificity": 0.6428571428571429,
    "precision": 0.4444444444444444,
    "ppv": 0.4444444444444444,
    "npv": 0.9,
    "f